<a href="https://www.kaggle.com/code/lelandhenderson/exploring-coffee-sales-using-a-decision-tree?scriptVersionId=327443650" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


/kaggle/input/exploring-coffee-sales-with-eda-and-visualization/Coffe_sales.csv


# PROBLEM DEFINED

The purpose of this analysis is to forcast the total money spent by a customer based on the independent factors:
1. hour_of_day: time that the customer entered
2. cash_type: method of payment used
3. money: total amount spent(dependent variable)
4. coffee_name: beverage purchased
5. Weekday
6. Date
7. Time

In [2]:
df = pd.read_csv('/kaggle/input/exploring-coffee-sales-with-eda-and-visualization/Coffe_sales.csv')

df.head()

,hour_of_day,cash_type,money,coffee_name,Time_of_Day,Weekday,Month_name,Weekdaysort,Monthsort,Date,Time
0,10,card,38.7,Latte,Morning,Fri,Mar,5,3,2024-03-01,10:15:50.520000
1,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:19:22.539000
2,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:20:18.089000
3,13,card,28.9,Americano,Afternoon,Fri,Mar,5,3,2024-03-01,13:46:33.006000
4,13,card,38.7,Latte,Afternoon,Fri,Mar,5,3,2024-03-01,13:48:14.626000


# EXPLORATORY DATA ANALYSIS

In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3547 entries, 0 to 3546
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   hour_of_day  3547 non-null   int64  
 1   cash_type    3547 non-null   object 
 2   money        3547 non-null   float64
 3   coffee_name  3547 non-null   object 
 4   Time_of_Day  3547 non-null   object 
 5   Weekday      3547 non-null   object 
 6   Month_name   3547 non-null   object 
 7   Weekdaysort  3547 non-null   int64  
 8   Monthsort    3547 non-null   int64  
 9   Date         3547 non-null   object 
 10  Time         3547 non-null   object 
dtypes: float64(1), int64(3), object(7)
memory usage: 304.9+ KB


In [4]:
df.isna().sum()

hour_of_day    0
cash_type      0
money          0
coffee_name    0
Time_of_Day    0
Weekday        0
Month_name     0
Weekdaysort    0
Monthsort      0
Date           0
Time           0
dtype: int64

In [5]:
df.describe()

,hour_of_day,money,Weekdaysort,Monthsort
count,3547.000000,3547.000000,3547.000000,3547.000000
mean,14.185791,31.645216,3.845785,6.453905
std,4.234010,4.877754,1.971501,3.500754
min,6.000000,18.120000,1.000000,1.000000
25%,10.000000,27.920000,2.000000,3.000000
50%,14.000000,32.820000,4.000000,7.000000
75%,18.000000,35.760000,6.000000,10.000000
max,22.000000,38.700000,7.000000,12.000000


In [6]:
df['Weekdaysort'].value_counts()

Weekdaysort
2    572
1    544
5    532
4    510
3    500
6    470
7    419
Name: count, dtype: int64

# FEATURE ENGINEERING

In [7]:
#convert to category: coffee_name, cash_type, Weekdaysort, Monthsort
df['coffee_name'] = df['coffee_name'].astype('category')
df['cash_type'] = df['cash_type'].astype('category')
df['Weekdaysort'] = df['Weekdaysort'].astype('category')
df['Monthsort'] = df['Monthsort'].astype('category')
# Deciscion Tree would work well with the high ratio of categorical data

In [8]:
df['coffee_name'].value_counts()


coffee_name
Americano with Milk    809
Latte                  757
Americano              564
Cappuccino             486
Cortado                287
Hot Chocolate          276
Cocoa                  239
Espresso               129
Name: count, dtype: int64

In [9]:
df['coffee_name'] = df['coffee_name'].map({'Americano with Milk':0,'Latte':1,'Americano':2,'Cappuccino':3,'Cortado':4,'Hot Chocolate':5,'Cocoa':6,'Expresso':7})
df = df.dropna(subset='coffee_name')
df.head()

,hour_of_day,cash_type,money,coffee_name,Time_of_Day,Weekday,Month_name,Weekdaysort,Monthsort,Date,Time
0,10,card,38.7,1.0,Morning,Fri,Mar,5,3,2024-03-01,10:15:50.520000
1,12,card,38.7,5.0,Afternoon,Fri,Mar,5,3,2024-03-01,12:19:22.539000
2,12,card,38.7,5.0,Afternoon,Fri,Mar,5,3,2024-03-01,12:20:18.089000
3,13,card,28.9,2.0,Afternoon,Fri,Mar,5,3,2024-03-01,13:46:33.006000
4,13,card,38.7,1.0,Afternoon,Fri,Mar,5,3,2024-03-01,13:48:14.626000


# TEST TRAIN SPLIT

In [10]:
from sklearn.model_selection import train_test_split
y = df['money']
X = df.drop(['money','Time_of_Day','Weekday','Month_name','Date','Time','cash_type'],axis=1)

X_train,X_test,y_train,y_test =train_test_split(X, y, test_size=.3, random_state=42)
X_train.isna().sum()

hour_of_day    0
coffee_name    0
Weekdaysort    0
Monthsort      0
dtype: int64

# DECISION TREE REGRESSOR

In [11]:
from sklearn.tree import DecisionTreeRegressor

tree=DecisionTreeRegressor(max_depth = 7,random_state=42)

In [12]:
tree.fit(X_train,y_train)

DecisionTreeRegressor(max_depth=7, random_state=42)

# TREE SCORING

In [13]:
print(tree.score(X_train,y_train))
print(tree.score(X_test,y_test))

0.9770348685655841
0.9700935895025757
